In [0]:
WITH metricas_por_canal AS (
    SELECT
        canal,
        SUM(CASE WHEN status_transacao = 'APROVADA' THEN quantidade_eventos ELSE 0 END) AS total_aprovadas,
        SUM(CASE WHEN status_transacao = 'ESTORNADA' THEN quantidade_eventos ELSE 0 END) AS total_estornadas,
        SUM(valor_total_liquido) AS volume_financeiro_liquido
    FROM novarota.gold.gold_desempenho_canais
    GROUP BY canal
)
SELECT
    canal,
    total_aprovadas,
    total_estornadas,
    ROUND(
        (total_estornadas * 100.0) / NULLIF((total_aprovadas + total_estornadas), 0), 
        2
    ) AS taxa_estorno_pct,
    volume_financeiro_liquido,
    ROUND(
        (volume_financeiro_liquido * 100.0) / SUM(volume_financeiro_liquido) OVER (),
        2
    ) AS share_volume_total_pct
FROM metricas_por_canal
ORDER BY volume_financeiro_liquido DESC